In [1]:
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

CUDA: True | Tesla T4


In [2]:
from google.colab import files
files.upload()
!unzip -q asl_project.zip -d /content
%cd /content/asl_project

Saving asl_project.zip to asl_project.zip
/content/asl_project


In [ ]:
!unzip -q asl_project.zip -d /content
!ls /content

asl_project  asl_project.zip  sample_data


In [3]:
!pwd
!ls

/content/asl_project
backend  collect_dataset.py  dataset  frontend	README.md  training


In [4]:
%cd /content/asl_project
!ls

/content/asl_project
backend  collect_dataset.py  dataset  frontend	README.md  training


In [5]:
import os
for c in sorted(os.listdir("dataset")):
    print(c, len(os.listdir(f"dataset/{c}")))

A 600
B 600
C 600
D 600
E 600
F 600
G 600
H 600
I 600
J 600
K 600
L 600
M 600
N 600
O 600
P 600
Q 600
R 600
S 600
T 600
U 600
V 600
W 600
X 600
Y 600
Z 600


In [6]:
!python training/split_data.py

wrote training/split.json


In [7]:
import os
import json
import time
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

DATASET = "dataset"
SPLIT = "training/split.json"

split = json.load(open(SPLIT))
classes = sorted(split["train"].keys())
idx = {c: i for i, c in enumerate(classes)}


def load(part):
    imgs, ys = [], []
    for c in classes:
        for f in split[part][c]:
            g = cv2.imread(os.path.join(DATASET, c, f), cv2.IMREAD_GRAYSCALE)
            imgs.append(g)
            ys.append(idx[c])
    return imgs, np.array(ys)


def otsu_feat(g):
    g = cv2.resize(g, (64, 64))
    _, t = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return t.flatten().astype("float32") / 255.0


def hog_feat(g):
    g = cv2.resize(g, (128, 128))
    return hog(g, orientations=9, pixels_per_cell=(16, 16),
               cells_per_block=(2, 2), feature_vector=True)


def feats(imgs, fn):
    return np.array([fn(im) for im in imgs])


def run(name, fn, model):
    Xtr = feats(tr_imgs, fn)
    Xte = feats(te_imgs, fn)
    t0 = time.time()
    model.fit(Xtr, ytr)
    train_t = time.time() - t0
    t0 = time.time()
    pred = model.predict(Xte)
    test_t = time.time() - t0
    acc = accuracy_score(yte, pred)
    f1 = f1_score(yte, pred, average="macro")
    print(f"{name}: acc={acc:.4f}  macroF1={f1:.4f}  "
          f"train={train_t:.1f}s  test={test_t:.2f}s")


tr_imgs, ytr = load("train")
te_imgs, yte = load("test")
print("classes:", len(classes), "train:", len(ytr), "test:", len(yte))

run("Baseline (Otsu + kNN, k=3)", otsu_feat, KNeighborsClassifier(n_neighbors=3))
run("Classical (HOG + SVM rbf, C=10)", hog_feat, SVC(kernel="rbf", C=10))

classes: 26 train: 6240 test: 780
Baseline (Otsu + kNN, k=3): acc=0.9974  macroF1=0.9974  train=0.0s  test=1.45s
Classical (HOG + SVM rbf, C=10): acc=1.0000  macroF1=1.0000  train=9.3s  test=2.85s


In [ ]:
!python training/train.py

saved preprocessing samples to training/preprocess_samples
epoch 1/15 val_acc 0.905
epoch 2/15 val_acc 0.931
epoch 3/15 val_acc 0.946
epoch 4/15 val_acc 0.985
epoch 5/15 val_acc 0.976
epoch 6/15 val_acc 0.987
epoch 7/15 val_acc 0.988
epoch 8/15 val_acc 0.988
epoch 9/15 val_acc 0.996
epoch 10/15 val_acc 0.988
epoch 11/15 val_acc 0.994
epoch 12/15 val_acc 0.987
epoch 13/15 val_acc 0.995
epoch 14/15 val_acc 0.996
epoch 15/15 val_acc 0.992
best val_acc 0.996 saved backend/model.pt


In [ ]:
!python training/evaluate.py

top-1 accuracy: 0.9936
macro-F1: 0.9935

              precision    recall  f1-score   support

           A      0.968     1.000     0.984        30
           B      1.000     1.000     1.000        30
           C      1.000     1.000     1.000        30
           D      1.000     1.000     1.000        30
           E      1.000     1.000     1.000        30
           F      1.000     1.000     1.000        30
           G      1.000     1.000     1.000        30
           H      1.000     1.000     1.000        30
           I      0.938     1.000     0.968        30
           J      1.000     1.000     1.000        30
           K      1.000     1.000     1.000        30
           L      1.000     0.900     0.947        30
           M      0.968     1.000     0.984        30
           N      1.000     0.967     0.983        30
           O      1.000     1.000     1.000        30
           P      1.000     1.000     1.000        30
           Q      1.000     1.000     1.

In [ ]:
from google.colab import files
import shutil
files.download("backend/model.pt")
files.download("backend/classes.json")
shutil.make_archive("preprocess_samples", "zip", "training/preprocess_samples")
files.download("preprocess_samples.zip")
files.download("training/confusion_matrix.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>